# Debs Ask: Documentary Interest -> Leads -> Sales (RFL Ratios)

- Start from Michael's market-level impressions
- Use simple ratios from RFL
- Downweight conversion assumptions because this is not a lower-funnel CTA campaign
- Estimate leads and sales by market and in total

No extra modeling beyond RFL ratios.

In [ ]:
from pathlib import Path
import pandas as pd
from openpyxl import load_workbook

BASE_DIR = Path('/home/ali/repos/porsche/house_of_porsche')
RFL_PATH = BASE_DIR / '2026_03_03_RevFLight_Simplified.xlsx'

# Michael's provided data (5-month proxy)
michael_data = pd.DataFrame(
    [
        ('Germany', 'PD', 493_400_000, 2.20),
        ('UK', 'PCGB', 155_100_000, 1.46),
        ('France', 'POF', 40_800_000, 4.06),
        ('US', 'PCNA', 573_700_000, 7.86),
        ('Canada', 'PCL', 24_100_000, 6.19),
    ],
    columns=['Market_Name', 'Market_Code', 'Impressions', 'CPM_EUR']
)

michael_data

,Market_Name,Market_Code,Impressions,CPM_EUR
0,Germany,PD,493400000,2.20
1,UK,PCGB,155100000,1.46
2,France,POF,40800000,4.06
3,US,PCNA,573700000,7.86
4,Canada,PCL,24100000,6.19


In [ ]:
# Read required RFL ratios by market code.
# Sheet used: Share Of MAL-Budget
# - Row 54: Average Impressions per Session
# - Row 28: Session to DCFS Rate
# - Row 25: DCFS to Sale Rate

wb = load_workbook(RFL_PATH, data_only=True)
ws = wb['Share Of MAL-Budget']

market_cols = {}
for c in range(2, ws.max_column + 1):
    code = ws.cell(3, c).value
    if isinstance(code, str) and code.strip() and code.strip().upper() in {'PD', 'PCGB', 'POF', 'PCNA', 'PCL'}:
        market_cols[code.strip().upper()] = c

ratio_rows = {
    'impressions_per_session': 54,
    'session_to_dcfs_rate': 28,
    'dcfs_to_sale_rate': 25,
}

ratio_records = []
for code, col in market_cols.items():
    ratio_records.append(
        {
            'Market_Code': code,
            'impressions_per_session': float(ws.cell(ratio_rows['impressions_per_session'], col).value or 0),
            'session_to_dcfs_rate': float(ws.cell(ratio_rows['session_to_dcfs_rate'], col).value or 0),
            'dcfs_to_sale_rate': float(ws.cell(ratio_rows['dcfs_to_sale_rate'], col).value or 0),
        }
    )

ratios = pd.DataFrame(ratio_records)
ratios

,Market_Code,impressions_per_session,session_to_dcfs_rate,dcfs_to_sale_rate
0,PCGB,22.50,0.00,0.14
1,PCL,26.22,0.00,0.01
2,PCNA,19.57,0.00,0.05
3,PD,19.11,0.00,0.11
4,POF,5.62,0.00,0.15


In [ ]:
# Debs asked to downweight conversion assumptions because this is not lower-funnel.
# Apply downweight only to session -> DCFS (DCFS -> sales remains unchanged).

import math

DOWNWEIGHT = 0.35  # 35% of lower-funnel efficiency for session -> DCFS only

est = michael_data.merge(ratios, on='Market_Code', how='left').copy()

# Pipeline: impressions -> sessions
est['estimated_sessions'] = est['Impressions'] / est['impressions_per_session']

# Without downweight
est['estimated_dcfs_no_downweight'] = est['estimated_sessions'] * est['session_to_dcfs_rate']
est['estimated_sales_no_downweight'] = est['estimated_dcfs_no_downweight'] * est['dcfs_to_sale_rate']

# With downweight (only session->DCFS)
est['effective_session_to_dcfs'] = est['session_to_dcfs_rate'] * DOWNWEIGHT
est['effective_dcfs_to_sale'] = est['dcfs_to_sale_rate']
est['estimated_dcfs_with_downweight'] = est['estimated_sessions'] * est['effective_session_to_dcfs']
est['estimated_sales_with_downweight'] = est['estimated_dcfs_with_downweight'] * est['effective_dcfs_to_sale']

# Ceiling requested for sessions / DCFS / sales.
for c in [
    'estimated_sessions',
    'estimated_dcfs_no_downweight', 'estimated_dcfs_with_downweight',
    'estimated_sales_no_downweight', 'estimated_sales_with_downweight'
]:
    est[c] = est[c].map(lambda v: math.ceil(v) if pd.notna(v) else v)

# Optional media spend read from Michael CPM for context
est['media_spend_eur'] = est['Impressions'] / 1000.0 * est['CPM_EUR']

cols = [
    'Market_Name', 'Market_Code', 'Impressions', 'media_spend_eur',
    'impressions_per_session',
    'session_to_dcfs_rate', 'effective_session_to_dcfs',
    'dcfs_to_sale_rate', 'effective_dcfs_to_sale',
    'estimated_sessions',
    'estimated_dcfs_no_downweight', 'estimated_dcfs_with_downweight',
    'estimated_sales_no_downweight', 'estimated_sales_with_downweight'
]

detail = est[cols].sort_values('Market_Name').copy()

# Force readable precision for tiny conversion rates.
for c in ['session_to_dcfs_rate', 'effective_session_to_dcfs', 'dcfs_to_sale_rate', 'effective_dcfs_to_sale']:
    detail[c] = detail[c].map(lambda v: f'{v:.6f}' if pd.notna(v) else '')

# Keep spend as rounded currency.
detail['media_spend_eur'] = detail['media_spend_eur'].map(lambda v: round(v, 2) if pd.notna(v) else v)

detail

,Market_Name,Market_Code,Impressions,media_spend_eur,impressions_per_session,session_to_dcfs_rate,effective_session_to_dcfs,dcfs_to_sale_rate,effective_dcfs_to_sale,estimated_sessions,estimated_dcfs_no_downweight,estimated_dcfs_with_downweight,estimated_sales_no_downweight,estimated_sales_with_downweight
4,Canada,PCL,24100000,"149,179.00",26.22,0.000418,0.000146,0.008723,0.008723,919300,385,135,4,2
2,France,POF,40800000,"165,648.00",5.62,0.000528,0.000185,0.145992,0.145992,7255152,3829,1340,559,196
0,Germany,PD,493400000,"1,085,480.00",19.11,0.000585,0.000205,0.112447,0.112447,25820100,15115,5290,1700,595
1,UK,PCGB,155100000,"226,446.00",22.50,0.001114,0.000390,0.144123,0.144123,6892073,7679,2688,1107,388
3,US,PCNA,573700000,"4,509,282.00",19.57,0.002402,0.000841,0.049761,0.049761,29317358,70430,24651,3505,1227


In [ ]:
# Executive summary output (what Ali can send to Debs)

total_impressions = est['Impressions'].sum()
total_spend = est['media_spend_eur'].sum()
total_sessions = est['estimated_sessions'].sum()
total_dcfs = est['estimated_dcfs_with_downweight'].sum()
total_sales = est['estimated_sales_with_downweight'].sum()

summary = pd.DataFrame(
    {
        'Metric': [
            'Total Impressions (Michael input)',
            'Estimated Media Spend (from CPM)',
            'Estimated Sessions (ceiling)',
            'Estimated Leads (DCFS, with downweight, ceiling)',
            'Estimated Sales (with downweight, ceiling)',
            'Downweight Applied to Session->DCFS',
        ],
        'Value': [
            f'{total_impressions:,.0f}',
            f'EUR {total_spend:,.0f}',
            f'{int(total_sessions):,}',
            f'{int(total_dcfs):,}',
            f'{int(total_sales):,}',
            f'{DOWNWEIGHT:.0%}',
        ],
    }
)

summary

,Metric,Value
0,Total Impressions (Michael input),"1,287,100,000"
1,Estimated Media Spend (from CPM),"EUR 6,136,035"
2,Estimated Sessions,"70,203,980"
3,"Estimated Leads (DCFS, with downweight)","34,102"
4,Estimated Sales (with downweight),842
5,Downweight Applied to Conversion Ratios,35%
